#  Feature Engineering
## Projet : Prédiction du Churn Client - Télécoms

### Objectif
Préparer les données pour le modèle Machine Learning :
1. **Encoder** les variables texte en chiffres
2. **Créer** de nouvelles variables intelligentes
3. **Sauvegarder** les données prêtes pour le modèle

### Pourquoi cette étape est importante ?
Un modèle ML ne comprend que les chiffres.
On doit donc transformer toutes les colonnes texte (Yes/No, Male/Female...)
en chiffres, et créer de nouvelles variables qui aident le modèle à mieux apprendre.

In [16]:
# =============================================================
# IMPORTS DES BIBLIOTHÈQUES
# =============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print(" Bibliothèques importées avec succès !")

 Bibliothèques importées avec succès !


##  1. Chargement des Données

On recharge les données brutes depuis le fichier CSV.
Toutes les corrections et transformations seront faites dans ce notebook.

In [17]:
# =============================================================
# CHARGEMENT DES DONNÉES
# =============================================================

# Chargement du fichier CSV
df = pd.read_csv('../data/raw/telco_churn.csv')

# Correction de TotalCharges (qu'on a découvert dans l'EDA)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Vérification
print(f" Dataset chargé : {df.shape[0]} clients, {df.shape[1]} variables")
print(f" TotalCharges type : {df['TotalCharges'].dtype}")
print(f"\n Aperçu des colonnes :")
print(df.dtypes)

 Dataset chargé : 7043 clients, 21 variables
 TotalCharges type : float64

 Aperçu des colonnes :
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object


##  2. Encodage des Variables

Un modèle ML ne comprend que les chiffres.
On doit convertir toutes les colonnes texte en chiffres.

**Deux types d'encodage :**
- **Binaire** : Yes/No → 1/0 (pour les colonnes à 2 valeurs)
- **One-Hot** : pour les colonnes avec plus de 2 valeurs
  (ex: Contract → Month-to-month, One year, Two year)
  On crée une colonne par valeur possible.

In [18]:
# =============================================================
# VALEURS DISTINCTES DE TOUTES LES COLONNES
# =============================================================

print(" Valeurs distinctes par colonne :")
print("=" * 50)

for col in df.columns:
    valeurs = df[col].unique()
    n = df[col].nunique()
    print(f"\n{col} ({n} valeurs distinctes) :")
    print(f"  {valeurs}")

 Valeurs distinctes par colonne :

customerID (7043 valeurs distinctes) :
  ['7590-VHVEG' '5575-GNVDE' '3668-QPYBK' ... '4801-JZAZL' '8361-LTMKD'
 '3186-AJIEK']

gender (2 valeurs distinctes) :
  ['Female' 'Male']

SeniorCitizen (2 valeurs distinctes) :
  [0 1]

Partner (2 valeurs distinctes) :
  ['Yes' 'No']

Dependents (2 valeurs distinctes) :
  ['No' 'Yes']

tenure (73 valeurs distinctes) :
  [ 1 34  2 45  8 22 10 28 62 13 16 58 49 25 69 52 71 21 12 30 47 72 17 27
  5 46 11 70 63 43 15 60 18 66  9  3 31 50 64 56  7 42 35 48 29 65 38 68
 32 55 37 36 41  6  4 33 67 23 57 61 14 20 53 40 59 24 44 19 54 51 26  0
 39]

PhoneService (2 valeurs distinctes) :
  ['No' 'Yes']

MultipleLines (3 valeurs distinctes) :
  ['No phone service' 'No' 'Yes']

InternetService (3 valeurs distinctes) :
  ['DSL' 'Fiber optic' 'No']

OnlineSecurity (3 valeurs distinctes) :
  ['No' 'Yes' 'No internet service']

OnlineBackup (3 valeurs distinctes) :
  ['Yes' 'No' 'No internet service']

DeviceProtection (3 val

In [19]:
# =============================================================
# ENCODAGE BINAIRE (Yes/No → 1/0)
# =============================================================

# Colonnes avec Yes/No
colonnes_binaires = ['Partner', 'Dependents', 'PhoneService', 
                     'PaperlessBilling', 'Churn']

for col in colonnes_binaires:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# Encodage du genre (Female/Male → 0/1)
df['gender'] = df['gender'].map({'Female': 0, 'Male': 1})

# Encodage SeniorCitizen (déjà en 0/1, rien à faire)

# Colonnes avec Yes/No/No service
colonnes_service = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                    'DeviceProtection', 'TechSupport', 
                    'StreamingTV', 'StreamingMovies']

for col in colonnes_service:
    df[col] = df[col].map({'Yes': 1, 'No': 0, 'No internet service': 0,
                           'No phone service': 0})

# Vérification
print(" Encodage binaire terminé !")
print("\nAperçu des colonnes encodées :")
print(df[['gender', 'Partner', 'Dependents', 'Churn'] + colonnes_service].head())

 Encodage binaire terminé !

Aperçu des colonnes encodées :
   gender  Partner  Dependents  Churn  MultipleLines  OnlineSecurity  \
0       0        1           0      0              0               0   
1       1        0           0      0              0               1   
2       1        0           0      1              0               1   
3       1        0           0      0              0               1   
4       0        0           0      1              0               0   

   OnlineBackup  DeviceProtection  TechSupport  StreamingTV  StreamingMovies  
0             1                 0            0            0                0  
1             0                 1            0            0                0  
2             1                 0            0            0                0  
3             0                 1            1            0                0  
4             0                 0            0            0                0  


##  3. One-Hot Encoding

Pour les colonnes avec plus de 2 valeurs, on utilise le One-Hot Encoding.
On crée une colonne par valeur possible.

**Exemple :**
Contract = "Month-to-month" → Contract_Month-to-month = 1, Contract_One year = 0, Contract_Two year = 0
Contract = "One year"       → Contract_Month-to-month = 0, Contract_One year = 1, Contract_Two year = 0

Les colonnes concernées :
- **Contract** : Month-to-month, One year, Two year
- **InternetService** : DSL, Fiber optic, No
- **PaymentMethod** : 4 méthodes de paiement

In [20]:
# =============================================================
# ONE-HOT ENCODING
# =============================================================

# Colonnes à encoder en One-Hot
colonnes_onehot = ['Contract', 'InternetService', 'PaymentMethod']

# On crée les colonnes One-Hot
# drop_first=True évite la multicolinéarité (on expliquera plus bas)
df = pd.get_dummies(df, columns=colonnes_onehot, drop_first=True)

# Supprimer customerID (inutile pour le modèle)
df = df.drop('customerID', axis=1)

# Vérification
print(f" One-Hot Encoding terminé !")
print(f" Nouvelles dimensions : {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(f"\n Liste de toutes les colonnes :")
for col in df.columns:
    print(f"  - {col}")

 One-Hot Encoding terminé !
 Nouvelles dimensions : 7043 lignes, 24 colonnes

 Liste de toutes les colonnes :
  - gender
  - SeniorCitizen
  - Partner
  - Dependents
  - tenure
  - PhoneService
  - MultipleLines
  - OnlineSecurity
  - OnlineBackup
  - DeviceProtection
  - TechSupport
  - StreamingTV
  - StreamingMovies
  - PaperlessBilling
  - MonthlyCharges
  - TotalCharges
  - Churn
  - Contract_One year
  - Contract_Two year
  - InternetService_Fiber optic
  - InternetService_No
  - PaymentMethod_Credit card (automatic)
  - PaymentMethod_Electronic check
  - PaymentMethod_Mailed check


On est passé de 21 colonnes à 24 colonnes et toutes sont maintenant numériques.
Pourquoi drop_first=True ?
Si Contract_Month-to-month = 0 ET Contract_One year = 0 alors on sait forcément que c'est Two year. Donc garder les 3 colonnes serait redondant. On en supprime toujours une.

##  4. Création de Nouvelles Variables

On crée de nouvelles variables à partir des variables existantes.
Ces nouvelles variables apportent des informations supplémentaires au modèle.

In [21]:
# =============================================================
# CRÉATION DE NOUVELLES VARIABLES
# =============================================================

# 1. Ratio charges mensuelles / ancienneté
# → Détecte si le client paye de plus en plus cher avec le temps
# → Un ratio élevé = client récent qui paye cher = risque de churn
df['charge_par_mois'] = df['MonthlyCharges'] / (df['tenure'] + 1)

# 2. Nombre de services souscrits
# → Un client avec beaucoup de services est plus engagé → moins à risque
services = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 
            'OnlineBackup', 'DeviceProtection', 'TechSupport',
            'StreamingTV', 'StreamingMovies']
df['nb_services'] = df[services].sum(axis=1)

# 3. Segment d'ancienneté
# → Catégoriser les clients par leur ancienneté
df['nouveau_client'] = (df['tenure'] <= 6).astype(int)      # 0-6 mois
df['client_moyen'] = ((df['tenure'] > 6) & 
                       (df['tenure'] <= 24)).astype(int)     # 6-24 mois
df['client_fidele'] = (df['tenure'] > 24).astype(int)       # + 24 mois

# 4. Charges totales estimées manquantes
# → Pour les nouveaux clients (tenure=0), on estime leurs charges futures
df['charges_estimees'] = df['MonthlyCharges'] * 12

# Vérification
print(" Nouvelles variables créées !")
print(f" Nouvelles dimensions : {df.shape[0]} lignes, {df.shape[1]} colonnes")
print("\n Aperçu des nouvelles variables :")
print(df[['tenure', 'MonthlyCharges', 'charge_par_mois', 
          'nb_services', 'nouveau_client', 
          'client_moyen', 'client_fidele']].head(10))

 Nouvelles variables créées !
 Nouvelles dimensions : 7043 lignes, 30 colonnes

 Aperçu des nouvelles variables :
   tenure  MonthlyCharges  charge_par_mois  nb_services  nouveau_client  \
0       1           29.85        14.925000            1               1   
1      34           56.95         1.627143            3               0   
2       2           53.85        17.950000            3               1   
3      45           42.30         0.919565            3               0   
4       2           70.70        23.566667            1               1   
5       8           99.65        11.072222            5               0   
6      22           89.10         3.873913            4               0   
7      10           29.75         2.704545            1               0   
8      28          104.80         3.613793            6               0   
9      62           56.15         0.891270            3               0   

   client_moyen  client_fidele  
0             0            

##  5. Nouvelles Variables Avancées

On ajoute des variables qui combinent plusieurs informations
pour mieux capturer le risque de churn.

- **risque_depart** : client récent ET sans engagement
- **charge_par_service** : prix payé par service souscrit
- **client_isole** : célibataire sans dépendants

In [22]:
# =============================================================
# NOUVELLES VARIABLES AVANCÉES
# =============================================================

# 1. Client à risque de départ
# Client récent (< 6 mois) ET sans engagement (month-to-month)
df['risque_depart'] = (
    (df['nouveau_client'] == 1) &
    (df['Contract_One year'] == 0) &
    (df['Contract_Two year'] == 0)
).astype(int)

# 2. Charge par service souscrit
# Détecte si le client paye cher par rapport aux services qu'il a
df['charge_par_service'] = df['MonthlyCharges'] / (df['nb_services'] + 1)

# 3. Client isolé
# Célibataire sans dépendants = moins d'attaches = plus à risque
df['client_isole'] = (
    (df['Partner'] == 0) &
    (df['Dependents'] == 0)
).astype(int)

# Vérification
print(" Nouvelles variables avancées créées !")
print(f"\n Dimensions : {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(f"\n Aperçu des nouvelles variables :")
print(df[['tenure', 'nouveau_client', 'risque_depart',
          'MonthlyCharges', 'nb_services', 'charge_par_service',
          'Partner', 'Dependents', 'client_isole']].head(10))

# Distribution des nouvelles variables
print(f"\n Distribution risque_depart :")
print(df['risque_depart'].value_counts())
print(f"\n Distribution client_isole :")
print(df['client_isole'].value_counts())

 Nouvelles variables avancées créées !

 Dimensions : 7043 lignes, 33 colonnes

 Aperçu des nouvelles variables :
   tenure  nouveau_client  risque_depart  MonthlyCharges  nb_services  \
0       1               1              1           29.85            1   
1      34               0              0           56.95            3   
2       2               1              1           53.85            3   
3      45               0              0           42.30            3   
4       2               1              1           70.70            1   
5       8               0              0           99.65            5   
6      22               0              0           89.10            4   
7      10               0              0           29.75            1   
8      28               0              0          104.80            6   
9      62               0              0           56.15            3   

   charge_par_service  Partner  Dependents  client_isole  
0           14.925000  

##  6. Sauvegarde des Données Préparées

On sauvegarde les données nettoyées et enrichies dans le dossier `data/processed/`.
Ce fichier sera utilisé directement pour entraîner notre modèle ML.

In [23]:
# =============================================================
# SAUVEGARDE DES DONNÉES PRÉPARÉES
# =============================================================

# Chemin de sauvegarde
chemin = '../data/processed/telco_churn_processed.csv'

# Sauvegarde
df.to_csv(chemin, index=False)

# Vérification
df_test = pd.read_csv(chemin)
print(" Données sauvegardées avec succès !")
print(f" Dimensions : {df_test.shape[0]} lignes, {df_test.shape[1]} colonnes")
print(f" Chemin : {chemin}")
print(f"\n Colonnes finales :")
for i, col in enumerate(df_test.columns, 1):
    print(f"  {i:02d}. {col}")

 Données sauvegardées avec succès !
 Dimensions : 7043 lignes, 33 colonnes
 Chemin : ../data/processed/telco_churn_processed.csv

 Colonnes finales :
  01. gender
  02. SeniorCitizen
  03. Partner
  04. Dependents
  05. tenure
  06. PhoneService
  07. MultipleLines
  08. OnlineSecurity
  09. OnlineBackup
  10. DeviceProtection
  11. TechSupport
  12. StreamingTV
  13. StreamingMovies
  14. PaperlessBilling
  15. MonthlyCharges
  16. TotalCharges
  17. Churn
  18. Contract_One year
  19. Contract_Two year
  20. InternetService_Fiber optic
  21. InternetService_No
  22. PaymentMethod_Credit card (automatic)
  23. PaymentMethod_Electronic check
  24. PaymentMethod_Mailed check
  25. charge_par_mois
  26. nb_services
  27. nouveau_client
  28. client_moyen
  29. client_fidele
  30. charges_estimees
  31. risque_depart
  32. charge_par_service
  33. client_isole
